# 에이전트의 도구 사용(Tool Use)

agent가 도구를 쓴다는 말은 단순히 함수를 호출한다는 뜻이 아니다. 질문을 보고 어떤 도구가 필요한지 선택하고, 구조화된 입력을 만들고, 실행 결과를 다시 reasoning 흐름에 연결하는 전체 패턴을 뜻한다. 이 노트북은 안전한 계산기, 데이터 집계 도구, 텍스트 검색 도구를 예시로 tool registry 패턴을 설명한다.

## 학습 목표
- 왜 `eval()` 대신 AST 기반 `SafeEvaluator`를 쓰는지 설명할 수 있다.
- calculator, date_parser, keyword_extractor, data_tool, search_tool의 역할 차이를 이해한다.
- `ToolRegistry`가 도구 추가 비용을 어떻게 낮추는지 설명할 수 있다.
- agent가 질문을 보고 도구를 "선택"한다는 것이 무엇인지 이해한다.


## 개념 설명

tool use는 agent가 문서 검색만으로 해결하기 어려운 부분을 외부 함수 호출로 보완하는 메커니즘이다. 예를 들어 날짜 차이 계산은 문서에서 날짜 두 개를 읽는 것과, 그 차이를 정확히 계산하는 것이 서로 다른 작업이다. 전자는 retrieval, 후자는 tool use에 가깝다.

**목적**
- tool use를 function call이 아니라 reasoning extension으로 이해한다.

**핵심 로직**
- 질문을 본다.
- 필요한 도구를 고른다.
- 구조화된 인자를 만든다.
- 도구 결과를 다시 answer synthesis에 반영한다.

**결과 해석 가이드**
- 이 notebook은 "무슨 도구가 있나"보다 "왜 registry와 selection이 필요한가"를 보여주는 데 초점이 있다.

**💡 면접 포인트**
- "Tool use는 LLM이 못하는 일을 대체하는 것이 아니라, LLM이 안전하게 위임해야 할 일을 분리하는 것"이라고 정리할 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

먼저 `ToolRegistry`를 포함한 확장 도구 계층을 불러온다. registry 패턴을 쓰는 이유는 새 도구를 추가할 때 기존 호출 코드를 크게 바꾸지 않기 위해서다. 도구가 늘어날수록 if/elif 체인을 늘리는 방식보다 registry가 훨씬 관리하기 쉽다.

**목적**
- 도구 집합을 registry로 관리하는 구조를 이해한다.

**핵심 로직**
- `build_default_registry()`가 calculator, data_tool, search_tool을 등록한다.
- `ToolCall`은 구조화된 호출 단위다.

**실제 소스 코드: ToolRegistry — src/tools_extended.py**
```python
class ToolRegistry:
    def __init__(self) -> None:
        self._handlers: dict[str, ToolHandler] = {}

    def register(self, name: str, handler: ToolHandler) -> None:
        self._handlers[name] = handler

    def list_tools(self) -> list[str]:
        return sorted(self._handlers)

    def call(self, tool_call: ToolCall) -> ToolResult:
        handler = self._handlers.get(tool_call.tool_name)
        if handler is None:
            return ToolResult(tool_call.tool_name, False, {"error": "Unknown tool"})
        output = handler(**tool_call.args)
        return ToolResult(tool_call.tool_name, bool(output.get("ok", False)), output)

    def select_tools(self, query: str) -> list[str]:
        normalized = normalize_text(query).lower()
        selected: list[str] = []
        if any(marker in normalized for marker in ("calculate", "sum", "difference", "days", "total")):
            selected.append("calculator")
        if any(marker in normalized for marker in ("average", "count", "dataset", "rows", "unique")):
            selected.append("data_tool")
        if any(marker in normalized for marker in ("search", "find", "lookup", "which document")):
            selected.append("search_tool")
        return selected or ["search_tool"]
```

**코드 읽기 포인트**
- `register(name, handler)`로 도구를 이름 기반으로 붙인다.
- `call()`은 `ToolCall`을 받아 `ToolResult`로 돌려주므로, 도구 성공/실패 형식이 통일된다.
- registry가 있으면 notebook, workflow, evaluator가 같은 tool interface를 재사용할 수 있다.

**결과 해석 가이드**
- registry가 list_tools로 노출되면 현재 실험 가능한 도구 집합을 즉시 확인할 수 있다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.tools_extended import ToolCall, build_default_registry

pd.set_option('display.max_colwidth', 140)
registry = build_default_registry()


## 도구 목록과 선택 기준

도구 목록을 보는 것만으로는 부족하다. agent는 질문을 보고 어떤 도구를 꺼낼지 스스로 판단해야 한다. 이 저장소에서는 `select_tools()`가 그 역할을 한다. 아직 단순 heuristic이지만, 바로 그렇기 때문에 어떤 marker가 어떤 tool을 부르는지 투명하다.

**목적**
- 질문-도구 매핑 규칙을 이해한다.

**핵심 로직**
- 질문에 계산 관련 marker가 있으면 calculator를, 데이터 집계 marker가 있으면 data_tool을, 검색 marker가 있으면 search_tool을 선택한다.
- 아무것도 안 맞으면 기본값으로 search_tool을 반환한다.

**실제 소스 코드: ToolRegistry.select_tools() — src/tools_extended.py**
```python
    def select_tools(self, query: str) -> list[str]:
        normalized = normalize_text(query).lower()
        selected: list[str] = []
        if any(marker in normalized for marker in ("calculate", "sum", "difference", "days", "total")):
            selected.append("calculator")
        if any(marker in normalized for marker in ("average", "count", "dataset", "rows", "unique")):
            selected.append("data_tool")
        if any(marker in normalized for marker in ("search", "find", "lookup", "which document")):
            selected.append("search_tool")
        return selected or ["search_tool"]
```

**코드 읽기 포인트**
- `calculate`, `days`, `total` 같은 표현이 calculator로 이어진다.
- `average`, `count`, `rows`는 data_tool 쪽으로 간다.
- `search`, `find`, `lookup`은 search_tool로 간다.
- 즉 "agent가 도구를 선택한다"는 것은 신비한 능력이 아니라, 질문 의도를 도구 affordance에 매핑하는 것이다.

**결과 해석 가이드**
- selected_tools가 비어 있지 않다면 최소한 하나의 해결 경로가 있다고 본다.
- 기본값이 search_tool이라는 것은, 텍스트 검색을 가장 일반적인 fallback tool로 본다는 뜻이다.


In [ ]:
registry.list_tools()


## 구조화된 도구 호출(Structured Tool Calls)

이제 실제 도구 구현을 본다. 특히 calculator는 `eval()` 대신 AST를 직접 순회하는 `SafeEvaluator`를 쓴다는 점이 중요하다. 사용자가 입력한 수식을 그대로 `eval()` 하면 임의 코드 실행 위험이 생기기 때문이다. 교육용 repo라도 이런 보안 기본기는 꼭 드러내야 한다.

**목적**
- 안전한 계산 도구와 기본 도구들의 역할을 코드 수준에서 이해한다.

**핵심 로직**
- `SafeEvaluator`는 허용된 AST node만 방문한다.
- `calculator()`는 AST parse 후 evaluator를 돌린다.
- `date_parser()`는 `dateutil`을 써 자연어 날짜를 정규화한다.
- `keyword_extractor()`는 핵심 토큰 빈도를 간단히 뽑는다.

**주요 파라미터/변수**
- `expression`: 계산할 수식
- `text`: 날짜 파싱/키워드 추출 대상 텍스트
- `limit`: 키워드 개수

**실제 소스 코드: SafeEvaluator — src/tools.py**
```python
class SafeEvaluator(ast.NodeVisitor):
    def visit_Expression(self, node: ast.Expression) -> float:
        return self.visit(node.body)

    def visit_BinOp(self, node: ast.BinOp) -> float:
        operator_fn = ALLOWED_BINARY_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError("Unsupported operation in calculator expression.")
        return operator_fn(self.visit(node.left), self.visit(node.right))

    def visit_UnaryOp(self, node: ast.UnaryOp) -> float:
        operator_fn = ALLOWED_UNARY_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError("Unsupported unary operation in calculator expression.")
        return operator_fn(self.visit(node.operand))

    def visit_Constant(self, node: ast.Constant) -> float:
        if not isinstance(node.value, int | float):
            raise ValueError("Calculator accepts numeric constants only.")
        return float(node.value)

    def generic_visit(self, node: ast.AST) -> float:
        raise ValueError(f"Unsupported calculator syntax: {type(node).__name__}")
```

**실제 소스 코드: calculator() — src/tools.py**
```python
def calculator(expression: str) -> dict[str, Any]:
    try:
        parsed = ast.parse(expression, mode="eval")
        result = SafeEvaluator().visit(parsed)
    except Exception as error:  # pragma: no cover - simple error normalization
        return {"ok": False, "error": str(error)}
    return {"ok": True, "result": int(result) if result.is_integer() else round(result, 3)}
```

**실제 소스 코드: date_parser() — src/tools.py**
```python
def date_parser(text: str, reference_date: str = DEFAULT_REFERENCE_DATE) -> dict[str, Any]:
    normalized = normalize_text(text).lower()
    anchor = datetime.fromisoformat(reference_date).date()

    if normalized == "last quarter":
        quarter = (anchor.month - 1) // 3
        year = anchor.year if quarter > 0 else anchor.year - 1
        quarter = quarter if quarter > 0 else 4
        start_month = (quarter - 1) * 3 + 1
        end_month = start_month + 2
        start = date(year, start_month, 1)
        end = date(year, end_month, calendar.monthrange(year, end_month)[1])
        return {"ok": True, "normalized": {"start": start.isoformat(), "end": end.isoformat()}}

    try:
        parsed = date_parser_lib.parse(text, fuzzy=True, default=datetime(anchor.year, 1, 1))
    except (ValueError, OverflowError):
        return {"ok": False, "error": f"Could not parse date from: {text}"}
    return {"ok": True, "normalized": parsed.date().isoformat()}
```

**실제 소스 코드: keyword_extractor() — src/tools.py**
```python
def keyword_extractor(text: str, limit: int = 5) -> dict[str, Any]:
    keywords = Counter(content_tokens(text))
    return {"ok": True, "keywords": [token for token, _ in keywords.most_common(limit)]}
```

**코드 읽기 포인트**
- `generic_visit()`에서 허용되지 않은 AST node를 즉시 막는 것이 보안 핵심이다.
- `calculator()`는 에러를 예외로 올리지 않고 `{ok: False, error: ...}` 형태로 정규화한다.
- `date_parser()`가 `dateutil`을 쓰는 이유는 사람이 쓰는 날짜 표현을 유연하게 읽기 위해서다.
- `keyword_extractor()`는 단순하지만 summary 보조 신호로 충분히 유용하다.

**결과 해석 가이드**
- structured result 표에서 `ok=True`면 도구가 정상 수행된 것이다.
- date parser 결과는 정규화된 ISO 문자열 형태라 이후 calculator나 verifier가 재사용하기 쉽다.

**💡 면접 포인트**
- "Tool layer에서는 성능보다 안전성과 형식 일관성이 더 중요하다"고 말할 수 있다.


In [ ]:
structured_calls = [
    ToolCall('calculator', {'expression': '12 * 3 + 4'}),
    ToolCall('data_tool', {'rows': [{'team': 'A', 'score': 8}, {'team': 'B', 'score': 10}, {'team': 'A', 'score': 6}], 'column': 'score', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'launch date', 'documents': ['The launch date is May 5, 2025.', 'The pilot begins in March.', 'Finance approved the budget.']}),
]
structured_results = [registry.call(tool_call).to_dict() for tool_call in structured_calls]
pd.DataFrame(structured_results)


## 도구 선택(tool selection)

선택 로직은 도구 자체만큼 중요하다. 도구가 많아질수록 잘못된 도구를 고르는 비용도 커지기 때문이다. 이 셀은 자연어 질문이 어떤 tool list로 매핑되는지 보여준다.

**목적**
- 질문 유형과 도구 선택 규칙의 대응을 본다.

**결과 해석 가이드**
- 질문에 계산 표현이 있는데 calculator가 빠지면 selection heuristic을 손봐야 한다.
- search_tool이 기본값으로 많이 나오는 것은 보수적이지만, 지나치면 불필요한 텍스트 검색이 늘 수 있다.


In [ ]:
selection_examples = [
    'Calculate the pilot duration in days.',
    'Find which document mentions the launch date.',
    'Count how many rows are in this dataset.',
]
pd.DataFrame(
    {
        'query': selection_examples,
        'selected_tools': [', '.join(registry.select_tools(query)) for query in selection_examples],
    }
)


## 실험

마지막으로 확장 도구인 `data_tool`과 `search_tool`까지 포함해 실제 호출을 돌려 본다. `data_tool`은 작은 구조화 데이터 집계에, `search_tool`은 여러 문장 중 관련 문장을 먼저 찾는 데 적합하다. 즉 도구마다 잘하는 질문 패턴이 다르다.

**목적**
- 도구별 적합한 사용처를 비교한다.

**핵심 로직**
- `data_tool()`은 DataFrame으로 변환한 뒤 count/unique/mean 연산을 제공한다.
- `search_tool()`은 여러 문장을 점수화해 top_k 결과를 돌려준다.

**실제 소스 코드: data_tool() — src/tools_extended.py**
```python
def data_tool(rows: list[dict[str, Any]], column: str, operation: str = "count") -> dict[str, Any]:
    frame = pd.DataFrame(rows)
    if column not in frame.columns:
        return {"ok": False, "error": f"Column '{column}' not found."}

    series = frame[column]
    if operation == "count":
        return {"ok": True, "operation": operation, "result": int(series.count())}
    if operation == "unique":
        return {"ok": True, "operation": operation, "result": sorted(series.dropna().astype(str).unique().tolist())}
    if operation == "mean":
        numeric = pd.to_numeric(series, errors="coerce")
        return {"ok": True, "operation": operation, "result": round(float(numeric.mean()), 3)}
    return {"ok": False, "error": f"Unsupported operation: {operation}"}
```

**실제 소스 코드: search_tool() — src/tools_extended.py**
```python
def search_tool(query: str, documents: list[str], top_k: int = 3) -> dict[str, Any]:
    query_tokens = content_tokens(query)
    ranked: list[dict[str, Any]] = []
    for index, document in enumerate(documents, start=1):
        score = round(overlap_ratio(query_tokens, content_tokens(document)), 4)
        ranked.append(
            {
                "document_id": index,
                "text": document,
                "score": score,
            }
        )
    results = sorted(ranked, key=lambda item: item["score"], reverse=True)[:top_k]
    return {"ok": True, "results": results}
```

**코드 읽기 포인트**
- `data_tool`은 숫자 집계처럼 구조가 명확한 입력에 강하다.
- `search_tool`은 retrieval-lite 성격이라, synthesis 전에 후보 텍스트를 줄이는 데 유용하다.
- 이 둘을 registry에 등록해 두면 planner나 workflow가 자연스럽게 재사용할 수 있다.

**결과 해석 가이드**
- 각 tool의 `result` 필드 구조가 다르므로, downstream synthesis는 tool type을 알고 읽어야 한다.
- 평균(mean) 같은 집계 결과가 정확하면 data_tool은 calculator보다 더 적합한 선택일 수 있다.


In [ ]:
experiment_calls = [
    ToolCall('calculator', {'expression': '25 - 7'}),
    ToolCall('data_tool', {'rows': [{'latency': 0.8}, {'latency': 1.2}, {'latency': 1.0}], 'column': 'latency', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'pilot window', 'documents': ['Pilot window: March 10, 2025 to April 4, 2025.', 'The governance memo explains ownership.', 'The FAQ lists tool guidance.']}),
]
experiment_results = [registry.call(tool_call).to_dict() for tool_call in experiment_calls]
pd.DataFrame(experiment_results)


## 결과 해석

마지막 표는 각 도구의 best-for를 요약한 것이다. 여기서 중요한 것은 "도구를 많이 갖는 것"보다 "질문 유형에 맞는 도구를 고를 수 있는 것"이다. tool registry와 selection 로직이 필요한 이유도 바로 여기에 있다.

**목적**
- 도구별 강점과 역할 경계를 정리한다.

**결과 해석 가이드**
- calculator는 정밀 계산, data_tool은 작은 표 집계, search_tool은 관련 텍스트 찾기에 강하다고 읽으면 된다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'tool': 'calculator', 'best_for': 'precise arithmetic and simple derived values'},
        {'tool': 'data_tool', 'best_for': 'small structured datasets and aggregates'},
        {'tool': 'search_tool', 'best_for': 'finding relevant text before synthesis'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 tool use는 단순 함수 호출이 아니라, 질문을 보고 적절한 외부 능력을 선택해 reasoning 흐름에 연결하는 과정이라는 점을 확인했다. `SafeEvaluator`가 `eval()` 대신 AST를 쓰는 이유는 보안 때문이고, `ToolRegistry`는 도구 추가 시 기존 코드를 덜 건드리게 해 주는 구조적 장치다. 또한 calculator, data_tool, search_tool은 각각 강한 질문 유형이 다르므로, selection 로직이 실제 품질을 크게 좌우한다.

**💡 면접 포인트**
- "Tool use는 LLM이 직접 계산·집계를 하지 않도록 역할을 분리하는 설계다."
- "AST 기반 calculator는 작은 구현이지만, 임의 코드 실행을 막는 중요한 보안 선택이다."
- "Registry 패턴을 쓰면 새로운 도구를 추가할 때 호출부 변경을 최소화할 수 있다."
